# Ringdown analysis with sky-location marginalization: Full-sky, Fixed-sky, Informed-sky

A short demonstration of `src/analysis.py` (the setup of the paper's runs) on GW250114 and GW190521.

- **Full-sky:** the sky position is sampled with an isotropic prior. All detectors are analysed from the arrival time
  at the last detector, and the other templates are shifted by the sky-dependent delay.
- **Fixed-sky:** the sky position is fixed at an IMR point estimate. Each detector is analysed from its own arrival time.
- **Informed-sky:** the Full-sky likelihood is averaged over 100 IMR sky samples, which act as the sky prior.

Each run uses `NLIVE` live points. The paper used $10^5$ for the Full-sky and Fixed-sky runs of Figs. 2–3, and
$10^4$ for Informed-sky and the start-time scans. With $10^4$ live points a Full-sky run takes about 35 minutes on a
laptop. The strain is read from `strain/` of the Zenodo data if `SKYRING_DATA` is set; otherwise it is downloaded from
GWOSC and cached in `./strain_cache`.

In [ ]:
import os
os.environ["JAX_PLATFORM_NAME"] = "cpu"
os.environ["XLA_FLAGS"] = "--xla_force_host_platform_device_count=12"   # CPU devices used by the sampler; before importing jax
import sys
sys.path.insert(0, '../src')
import numpy as onp
import analysis as an

DATA = os.environ.get("SKYRING_DATA")                     # unzipped Zenodo data (optional)
STRAIN = os.path.join(DATA, "strain") if DATA else "./strain_cache"
NLIVE = int(os.environ.get("SKYRING_NLIVE", 10_000))
MODES = {"GW250114": ((2, 2, 0), (2, 2, 1)), "GW190521": ((2, 2, 0), (3, 3, 0))}

def run(setup, modes, seed=31567, **kw):
    """Nested sampling of one model; returns the posterior samples, their names and ln Z."""
    names, low, high = an.prior_box(setup, modes, **{k: v for k, v in kw.items() if k in ("mass", "sky_frame")})
    loglik = an.make_loglik(setup, modes, **{k: v for k, v in kw.items() if k == "sky_frame"})
    res = an.run_nested(loglik, low, high, nlive=NLIVE, seed=seed, verbose=False)
    print(f"  {'+'.join(''.join(map(str, m)) for m in modes):8s} ln Z = {res['log_Z']:.2f} +- {res['log_Z_uncert']:.2f}"
          f" ({'converged' if res['converged'] else 'NOT converged'})")
    return res | {"names": names}

def bayes_factor(setup, modes, **kw):
    """ln B of the two-mode model over the (2,2,0)-only model."""
    full, single = run(setup, modes, **kw), run(setup, modes[:1], **kw)
    print(f"  ln B = {full['log_Z'] - single['log_Z']:.2f}")
    return full

In [ ]:
## GW250114, Full-sky (sky sampled; common window at the Hanford arrival)
setup_full = an.setup_event("GW250114", sky="full", cache_dir=STRAIN)
post_full = bayes_factor(setup_full, MODES["GW250114"])
A = dict(zip(post_full["names"], post_full["theta"].T))
print(f"  A221/A220 median {onp.median(A['A221'] / A['A220']):.2f}")

In [ ]:
## GW250114, Fixed-sky (sky fixed at (2.33, 0.19); each detector from its own arrival time)
setup_fixed = an.setup_event("GW250114", sky="fixed", cache_dir=STRAIN)
post_fixed = bayes_factor(setup_fixed, MODES["GW250114"])

In [ ]:
## GW250114, Informed-sky: Full-sky likelihood averaged over 100 IMR sky samples (equal weights)
## IMR sky samples: imr_sky/ of the Zenodo data, or (ra, dec) from the LVK GWTC-5 release (C00:NRSur7dq4)
sky = onp.load(os.path.join(DATA, "imr_sky", "GW250114_C00_NRSur7dq4_sky.npz"))
nodes, logw = an.imr_sky_nodes(sky["ra"], sky["dec"])
for modes in (MODES["GW250114"], MODES["GW250114"][:1]):
    names, low, high = an.prior_box(setup_full, modes, informed=True)
    res = an.run_nested(an.make_informed_loglik(setup_full, modes, nodes, logw), low, high, nlive=NLIVE,
                        seed=31567 ^ 0xABCDEF, verbose=False)        # seed of the paper's Informed-sky run
    print(f"  {'+'.join(''.join(map(str, m)) for m in modes):8s} ln Z = {res['log_Z']:.2f} +- {res['log_Z_uncert']:.2f}")

In [ ]:
## GW190521, Full-sky, (2,2,0) + (3,3,0) with A330 <= A220 (A220 in [0, 4.5], A330 in [0, 0.6], in units of 1e-20)
setup_190521 = an.setup_event("GW190521", sky="full", cache_dir=STRAIN)
post_190521 = bayes_factor(setup_190521, MODES["GW190521"])

In [ ]:
## Later start time (Figs. 4b, 5b): `shift` steps of t_M after the paper's start time; the scans used M >= 30 Msun
setup_shift = an.setup_event("GW250114", sky="full", shift=4, cache_dir=STRAIN)     # t_peak + 10 t_M
post_shift = bayes_factor(setup_shift, MODES["GW250114"], mass=(30., 95.))

In [ ]:
## Full-sky in detector-frame sky coordinates (epsilon, cos kappa about the H1-L1 baseline); same likelihood
post_det = run(setup_full, MODES["GW250114"], sky_frame="detector")